# Futbol Veri Projesi 1: Maç Verisini Toplamak ve Temizlemek

[![Colab'da Aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haticeozbolat01/veri-projeleri/blob/main/01-futbol-mac-verisi-toplama-temizleme.ipynb)

Bu not defteri, haticeozbolat.com'daki **Futbol Veri Projesi 1** yazısının kodlarını içeriyor. Yazı: [Yazıyı oku]([YAZI LİNKİ])

**Veri kaynağı:** [openfootball/football.json](https://github.com/openfootball/football.json), CC0 1.0 (kamu malı) lisanslı, gönüllülerin derlediği açık futbol verisi. Resmî bir kaynak değil; eksik ya da hatalı kayıtlar olabilir.

Hücreleri sırayla **Shift + Enter** ile çalıştır.

## 1. Veriyi indirmek

In [ ]:
import requests

URL = "https://raw.githubusercontent.com/openfootball/football.json/master/2024-25/tr.1.json"
cevap = requests.get(URL, timeout=30)
cevap.raise_for_status()
ham = cevap.json()

print(ham["name"])
print(len(ham["matches"]), "maç")

In [ ]:
ham["matches"][0]

## 2. JSON'dan tabloya

In [ ]:
import pandas as pd
import numpy as np

df = pd.json_normalize(ham["matches"])
print(df.shape)
df.head()

## 3. Sütunları düzenlemek

In [ ]:
df = df.rename(columns={
    "round": "hafta", "team1": "ev", "team2": "deplasman",
    "score.ft": "skor", "score.ht": "ilk_yari", "status": "durum",
})

df["hafta"] = df["hafta"].str.extract(r"(\d+)").astype(int)
df["tarih"] = pd.to_datetime(df["date"] + " " + df["time"])

def ayir(skor, i):
    """[ev, deplasman] listesinden i. elemanı alır; skor yoksa NaN döndürür."""
    return skor[i] if isinstance(skor, list) else np.nan

df["ev_gol"] = df["skor"].apply(ayir, i=0)
df["dep_gol"] = df["skor"].apply(ayir, i=1)
df["iy_ev"] = df["ilk_yari"].apply(ayir, i=0)
df["iy_dep"] = df["ilk_yari"].apply(ayir, i=1)
df["hukmen"] = df["durum"].eq("awarded")

df = df[["hafta", "tarih", "ev", "deplasman", "ev_gol", "dep_gol", "iy_ev", "iy_dep", "hukmen"]]
df.head()

## 4. Eksik değerler

In [ ]:
df.isna().sum()

In [ ]:
df[df["ev_gol"].isna()]

In [ ]:
df[df["hukmen"]]

## 5. Tutarlılık kontrolleri

In [ ]:
print("Haftadaki maç sayıları:", df.groupby("hafta").size().unique())
print("Takım sayısı:", pd.concat([df["ev"], df["deplasman"]]).nunique())
print("Takım başına maç:", pd.concat([df["ev"], df["deplasman"]]).value_counts().unique())
print("Aynı ev-deplasman eşleşmesi en fazla:", df.groupby(["ev", "deplasman"]).size().max(), "kez")

## 6. Oynanan maçlar ve yeni sütunlar

In [ ]:
oynanan = df.dropna(subset=["ev_gol", "dep_gol"]).copy()
oynanan[["ev_gol", "dep_gol"]] = oynanan[["ev_gol", "dep_gol"]].astype(int)

oynanan["toplam_gol"] = oynanan["ev_gol"] + oynanan["dep_gol"]
oynanan["sonuc"] = np.select(
    [oynanan["ev_gol"] > oynanan["dep_gol"], oynanan["ev_gol"] < oynanan["dep_gol"]],
    ["Ev sahibi", "Deplasman"],
    default="Beraberlik",
)
print(len(oynanan), "maç")
oynanan.head()

## 7. Temiz veriyi kaydetmek

In [ ]:
oynanan.to_csv("super_lig_2024_25_temiz.csv", index=False)
print("Kaydedildi.")

## 8. İlk bakış

In [ ]:
sahada = oynanan[~oynanan["hukmen"]]

print(f"Sahada oynanan maç: {len(sahada)}")
print(f"Maç başına gol: {sahada['toplam_gol'].mean():.2f}")
print(f"Ev sahibi gol ortalaması: {sahada['ev_gol'].mean():.2f}")
print(f"Deplasman gol ortalaması: {sahada['dep_gol'].mean():.2f}")
print()
print(sahada["sonuc"].value_counts(normalize=True).round(3))

In [ ]:
sahada.sort_values("toplam_gol", ascending=False).head(3)[["hafta", "ev", "deplasman", "ev_gol", "dep_gol"]]

In [ ]:
import matplotlib.pyplot as plt

sira = ["Ev sahibi", "Beraberlik", "Deplasman"]
sayilar = sahada["sonuc"].value_counts().reindex(sira)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sira, sayilar, color="#7A4290", width=0.6)
for x, s in enumerate(sayilar):
    ax.text(x, s + 3, f"{s} (%{s / sayilar.sum() * 100:.0f})", ha="center", color="#333333")
ax.set_ylabel("Maç sayısı")
ax.set_ylim(0, sayilar.max() * 1.18)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Süper Lig 2024-25: maç sonuçları (sahada oynanan maçlar)", loc="left")
plt.tight_layout()
plt.show()

## Kendin dene

1. URL'deki `tr.1` yerine `en.1` yazarak aynı sezonun Premier Lig verisini indir ve aynı temizliği uygula.
2. İlk yarı skoru olan maçlarda, ilk yarıyı önde kapatan takım maçı kaç kez kazanmış?
3. Haftalara göre maç başına gol ortalamasını hesapla. Sezon ilerledikçe değişiyor mu?
4. Ev sahibi kazanma oranı en yüksek olan takım hangisi?

## Sıradaki bölüm

**Futbol Veri Projesi 2:** Temiz veriden puan tablosu, ev sahibi avantajı ve takım analizi. [YAZI LİNKİ]